# async와 await

코루틴과 태스크의 실행 순서를 관찰하고 이벤트 루프를 막는 동기 대기를 안전하게 옮깁니다.

## 실행 환경 준비

Python 3.12 표준 라이브러리와 Jupyter의 최상위 `await`를 사용합니다. 이 절은 A·B 경로가 다시 합쳐지는 필수 절입니다. A 경로의 01~08절 가운데 특히 04절의 함수 호출·반환과 07절의 타입힌트를 사용합니다. B 경로의 09절 데코레이터와 `with`, 10절 Pydantic, 11절 fixture와 패키징은 필요하지 않습니다.

In [1]:
# 비동기 실행 순서와 시간을 관찰할 표준 도구를 가져옵니다.
import asyncio
import time
print("asyncio 준비 완료")

asyncio 준비 완료


## 기본 실행: 코루틴 만들고 기다리기

`async def fetch_label(name: str, delay: float) -> str`는 문자열 이름과 초 단위 실수 대기 시간을 입력받아 최종적으로 문자열을 반환하는 비동기 함수를 정의합니다. 이 타입힌트 표기는 07절에서 배웠습니다. 호출 직후에는 본문 결과가 아니라 코루틴 객체가 반환됩니다. Jupyter 셀의 `result = await pending`은 코루틴이 아직 완료되지 않았다면 기다리는 동안 다른 작업에 실행 기회를 주고, 완료되면 최종 문자열을 받습니다.

In [2]:
# Jupyter 셀에서 코루틴을 만들고 최상위 await로 기다립니다.
async def fetch_label(name: str, delay: float) -> str:
    # 실제 네트워크 대신 지정한 시간만 기다리는 모의 I/O입니다.
    await asyncio.sleep(delay)
    return f"완료:{name}"

pending = fetch_label("문서-A", 0.01)
print(type(pending).__name__)
result = await pending
print(result)

coroutine
완료:문서-A


## 태스크와 gather: 완료 순서와 반환 순서

이벤트 루프는 대기 중인 코루틴 사이에서 실행 순서를 조정합니다. 태스크는 코루틴을 이벤트 루프에 예약한 객체입니다. `tracked()`는 세 번째 인자로 받은 `progress` 목록에 완료된 이름을 추가합니다. 함수 밖에서 만든 목록도 함수 인자로 전달하면 같은 목록 객체를 수정할 수 있습니다. `gather`는 코루틴들을 함께 진행하고 모두 끝난 뒤 입력 순서의 결과 목록을 반환합니다.

In [3]:
# 완료 순서와 gather가 반환하는 입력 순서를 따로 기록합니다.
progress: list[str] = []

async def tracked(name: str, delay: float, progress: list[str]) -> str:
    await asyncio.sleep(delay)
    progress.append(name)
    return name

gathered = await asyncio.gather(
    tracked("느린 작업", 0.03, progress),
    tracked("빠른 작업", 0.01, progress),
)
print("완료 순서:", progress)
print("반환 순서:", gathered)
assert progress == ["빠른 작업", "느린 작업"]
assert gathered == ["느린 작업", "빠른 작업"]

완료 순서: ['빠른 작업', '느린 작업']
반환 순서: ['느린 작업', '빠른 작업']


## 순차 대기와 동시 대기 시간 비교

첫 측정은 `await fetch_label(...)`을 두 번 차례로 호출하므로 첫 반환 뒤에 둘째 호출이 시작됩니다. 다음 측정은 두 코루틴을 한 번의 `gather`에 입력하므로 두 대기가 함께 진행됩니다. 시간은 현재 컴퓨터에서 측정한 값이며 보편적인 속도 향상 비율을 뜻하지 않습니다.

In [4]:
# 같은 모의 I/O 지연을 순차 실행과 동시 실행으로 측정합니다.
start = time.perf_counter()
await fetch_label("A", 0.04)
await fetch_label("B", 0.04)
sequential = time.perf_counter() - start

start = time.perf_counter()
await asyncio.gather(
    fetch_label("A", 0.04), fetch_label("B", 0.04)
)
concurrent = time.perf_counter() - start
print(f"순차 {sequential:.3f}초 / 동시 {concurrent:.3f}초")

순차 0.083초 / 동시 0.042초


## Before: 동기 sleep이 이벤트 루프를 막음

`asyncio.Event()`는 코루틴 사이에서 한 사건의 발생 여부를 알리는 신호 객체입니다. `started.set()`은 신호를 켜고, `await started.wait()`는 신호가 켜질 때까지 기다립니다. `create_task(record_heartbeat(events, started))`가 하트비트를 예약한 뒤 시작 신호를 확인합니다. 이어서 동기 `time.sleep`이 이벤트 루프를 실행하는 한 작업 흐름을 점유하므로 하트비트는 이 호출이 반환된 뒤에야 이어집니다.

In [5]:
# 하트비트는 필요한 목록과 신호를 인자로 받아 사용합니다.
async def record_heartbeat(events, started):
    started.set()
    await asyncio.sleep(0.01)
    events.append("heartbeat")

# 시작 신호 뒤 동기 sleep이 하트비트를 지연시키는 순서를 봅니다.
async def blocked_probe() -> list[str]:
    events: list[str] = []
    started = asyncio.Event()
    task = asyncio.create_task(record_heartbeat(events, started))
    await started.wait()
    events.append("block-start")
    time.sleep(0.05)
    events.append("block-end")
    await task
    return events

blocked_events = await blocked_probe()
print(blocked_events)
assert blocked_events.index("heartbeat") > blocked_events.index("block-end")

['block-start', 'block-end', 'heartbeat']


## After: to_thread로 동기 작업을 이벤트 루프 밖에서 실행함

스레드는 한 프로세스 안에서 실행 흐름을 나누는 단위입니다. `asyncio.to_thread(time.sleep, 0.05)`는 동기 함수와 인자를 이벤트 루프와 다른 스레드에서 호출합니다. 이 호출이 완료되지 않은 동안 현재 코루틴이 기다리면 하트비트 태스크가 실행될 수 있습니다. 출력의 `True`는 이번 실행에서 `heartbeat`가 `wait-end`보다 먼저 기록됐다는 관찰입니다. 마지막 `assert`는 순서를 강제하지 않고 세 사건이 모두 기록됐는지만 검사합니다.

In [6]:
# 같은 하트비트 함수에 목록과 신호를 전달해 관찰 순서를 비교합니다.
async def responsive_probe() -> list[str]:
    events: list[str] = []
    started = asyncio.Event()
    task = asyncio.create_task(record_heartbeat(events, started))
    await started.wait()
    events.append("wait-start")
    await asyncio.to_thread(time.sleep, 0.05)
    events.append("wait-end")
    await task
    return events

responsive_events = await responsive_probe()
print(responsive_events)
print("하트비트가 먼저:", responsive_events.index("heartbeat") < responsive_events.index("wait-end"))
assert set(responsive_events) == {"wait-start", "heartbeat", "wait-end"}

['wait-start', 'heartbeat', 'wait-end']
하트비트가 먼저: True
